## Expectations — declarative data quality (+ `CHECK` on plain Delta)

**Expectations** are predicates you attach to a pipeline dataset. When a row violates the predicate, the pipeline reacts based on which keyword you used:

| Keyword | On violation |
|---|---|
| `expect` | Row **kept**; violation **logged** to the event log |
| `expect or drop` | Row **dropped**; violation logged |
| `expect or fail` | Pipeline **fails** on the first violation |

```python
@dlt.table
@dlt.expect_all({"valid_customer": "customer_id IS NOT NULL"})
@dlt.expect_or_drop("reasonable_amount", "amount < 1000000")
@dlt.expect_or_fail("valid_currency", "currency IN ('INR','USD','EUR')")
def silver_card_transactions():
    return dlt.read_stream("bronze_card_transactions")
```

Map severity to keyword: *track but don't block* → `expect`; *drop bad rows, keep running* → `expect or drop`; *stop and page a human* → `expect or fail`.

**`CHECK` constraints — quality without a pipeline.** Plain Delta tables enforce predicates on every write:

```sql
ALTER TABLE fintech_dev.silver.card_transactions
  ADD CONSTRAINT positive_amount CHECK (amount > 0);
```

The difference the exam contrasts: a `CHECK` constraint **fails the write** — there's no drop-and-log option — and it's table-level, applying to *every* writer. So: *"reject bad rows at write time on a plain Delta table"* → `CHECK`; *"track / drop / fail by severity in a pipeline"* → **expectations**.
